In [1]:
import polars as pl

## Creating the toy dataset

In [2]:
df = pl.scan_parquet('/home/ma/a/alb25/Project/thesis_code/data/processed/train_df.parquet')

In [4]:
source_users = df.select('source_user@domain').unique().collect()

In [9]:
toy_dataset_users = source_users.sample(n=1000)

In [14]:
df = df.select('time', 'source_user@domain').collect()

In [16]:
toy_dataset = df.join(toy_dataset_users, on='source_user@domain', how='semi')

In [ ]:
toy_dataset = toy_dataset.sort(['source_user@domain', 'time'])
toy_dataset.write_parquet(f'/home/ma/a/alb25/Project/thesis_code/data/processed/intermediate/toy_dataset.parquet',
                compression="lz4", statistics=True, row_group_size=250_000)

## Running lambert liu on the toy dataset

In [51]:
df = pl.read_parquet('/home/ma/a/alb25/Project/thesis_code/data/processed/intermediate/toy_dataset.parquet')

In [ ]:
def create_bins_and_counts(df, param_bin_width, evnt_bin_width):
    '''
    df : The dataframe to augment
    param_bin_width : parameter bin width is the bin width used for calculating the mean and variance lambert liu parameters
    evnt_bin_width : event bin width is the bin width used for grouping oberservations into events 

    returns a df with new columns
        param_bin_id
        param_bin_class
        param_bin_cnt

        envt_bin_id
        evnt_bin_class
        evnt_bin_count
    '''

    #### Adding parameter bin columns
    param_bins_per_week = int(24*60*7/param_bin_width)

    # Creating a bin id column for parameter bins
    df = df.with_columns((pl.col('time').dt.total_minutes()//param_bin_width).alias('param_bin_id'))
    # Creating a bin class which is the same for the same bin on different days i.e. 2 - 2:20 pm on different days will have the same 
    df = df.with_columns((pl.col('param_bin_id') % param_bins_per_week).alias('param_bin_class'))


    #### Adding event bin column
    # Creating a bin id column for evemt bins
    df = df.with_columns((pl.col('time').dt.total_minutes()//evnt_bin_width).alias('evnt_bin_id'))
    df = df.group_by(['evnt_bin_id', 'source_user@domain', 'param_bin_class']).agg(pl.len().alias('evnt_bin_count'))

    return df



def add_num_bins(df, param_bin_width, evnt_bin_width):
    '''
    Calculates how many event bins we have in our dataset 
    Right now we have empty bins as missing rows this creates a df of all the event bins possible
    and assigns them to a parameter bin 
    '''

    evnt_bins_per_param_bin = param_bin_width/evnt_bin_width
    param_bins_per_week = 7*24*60/param_bin_width

    total_evnt_bins = df.select(pl.col('evnt_bin_id').max()).item() + 1


    bins_df = pl.DataFrame({'evnt_bin_id': range(total_evnt_bins)})

    # Assigning to param bins
    bins_df = bins_df.with_columns(
        (pl.col('evnt_bin_id')//evnt_bins_per_param_bin).alias('param_bin_id'))

    # Assigning each param bin to param class
    bins_df = bins_df.with_columns(
        (pl.col('param_bin_id') % param_bins_per_week).alias('param_bin_class'))

    bins_df = bins_df.group_by('param_bin_class').agg(
        pl.col('evnt_bin_id').n_unique().alias('n_evnt_bins'))
    
    df = df.join(bins_df, on='param_bin_class', how='left')

    return df


def calc_mu_lambda(df, param_bin_width, evnt_bin_width):
    ''' 
    Takes a df assigns all observations to event and parameter bins and calculates lambda and mu
    '''

    # Creat bins and find how many events happen in each bin
    df = create_bins_and_counts(df, param_bin_width, evnt_bin_width)

    # Calculate the mean and squared mean of observation in the bin
    df = df.group_by(['source_user@domain', 'param_bin_class']).agg(
        pl.col('evnt_bin_count').sum().alias('sigma_a'),
        (pl.col('evnt_bin_count')**2).sum().alias('sigma_a2'))

    # For each parameter bin add how many events were used in these sigma a and sigma a^2 stas
    df = add_num_bins(df, param_bin_width, evnt_bin_width)

    # Calculate the mean and sample variance inside each event bin
    df = df.with_columns(
        mu = (pl.col('sigma_a') / pl.col('n_evnt_bins')),
        var = (pl.col('sigma_a2') - (pl.col('sigma_a')**2/pl.col('n_evnt_bins')))/ (pl.col('n_evnt_bins') - 1))
    
    return df 

evnt_bin_id,source_user@domain,param_bin_class,evnt_bin_count
i64,str,i64,u32
467,"""C1667$@DOM1""",77,4
1745,"""C4347$@DOM1""",122,4
1238,"""C21309$@DOM1""",38,4
4372,"""C22454$@DOM1""",56,11
4845,"""C3265$@DOM1""",135,4
…,…,…,…
2110,"""U3942@DOM1""",15,6
3115,"""C20754$@DOM1""",15,2
4871,"""C559$@DOM1""",139,4


In [ ]:
# 222
df.group_by(['source_user@domain', 'param_bin_class']).agg(
    pl.col('evnt_bin_id').n_unique().alias('uev')
).select('uev').max()

uev
u32
222


In [ ]:
## We need to get the small bin counts then big bin parameter estimates
# We need to calculate lambda and mu seperately


In [ ]:

##!! network service is stil in my toy dataset